# Fine-tuning Traffic Vision on road data

Your served model is COCO-pretrained. COCO is general photography — it has never
seen a traffic camera's angle, its lighting, or its weather. Measured on the
traffic classes, `car` recall sits at **0.255**: the model misses roughly three
out of four cars. Fine-tuning on actual road imagery is the only remaining fix.

**Run this on Colab, not your laptop.** Your machine is CPU-only; a real run
would take days there and about an hour here.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. If you skip
this, the next cell will tell you.

## 1. Confirm you actually have a GPU

If this prints `cpu`, stop and switch the runtime. Everything below assumes a GPU.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"


## 2. Install

Colab ships an older ultralytics; pin the version the project serves so the weights you produce load without surprises.

In [ ]:
!pip install -q ultralytics==8.4.120
import ultralytics; ultralytics.checks()


## 3. Get a labelled traffic dataset

This is the part that decides whether the fine-tune is worth anything. Four
realistic options:

| Dataset | Size | Why you would pick it | Friction |
|---|---|---|---|
| **Roboflow Universe** | varies | Fastest start. Exports YOLO format directly, free API key | Quality varies wildly — check the class balance before trusting it |
| **BDD100K** | 100k images | Best for your night problem — images carry explicit day/night attributes | Registration, ~7 GB |
| **Cityscapes** | 5k fine | Only one here with an on-rails class, i.e. **trams and street metro** | Registration, academic-ish |
| **KITTI** | 7.5k | Classic, clean labels, well documented | Registration, daytime only, German roads |

Start with Roboflow to get the pipeline working, then move to BDD100K for the
night gap or Cityscapes if trams matter to you.

Get a free key at roboflow.com → Settings → API Key, then browse
universe.roboflow.com for a traffic/vehicle detection dataset and copy its
workspace/project/version from the download snippet.

In [ ]:
# --- Option A: Roboflow (fastest to a working pipeline) -------------------
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# ds = rf.workspace("WORKSPACE").project("PROJECT").version(1).download("yolov11")
# DATA_YAML = f"{ds.location}/data.yaml"

# --- Option B: smoke test only -------------------------------------------
# WARNING: coco128 evaluates on its own training images -- train and val are
# the same 128 files. It will report huge gains that are pure memorisation.
# Use it to confirm the pipeline runs, then DELETE this line and use Option A.
DATA_YAML = "coco128.yaml"

print("dataset config:", DATA_YAML)


### Check the data before you train on it

The most expensive mistake is training for an hour on a broken dataset. Two
failures account for most of it: labels not mirroring images, and a class
distribution so skewed that a class cannot be learned.

In [ ]:
from pathlib import Path
import yaml, collections

if DATA_YAML != "coco128.yaml":
    cfg = yaml.safe_load(open(DATA_YAML))
    root = Path(DATA_YAML).parent
    print("classes:", cfg["names"], "\n")

    for split in ("train", "val"):
        if split not in cfg:
            print(f"{split}: MISSING from data.yaml"); continue
        img_dir = (root / cfg[split]).resolve()
        lbl_dir = Path(str(img_dir).replace("images", "labels"))
        imgs = list(img_dir.glob("*.*")) if img_dir.exists() else []
        lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
        print(f"{split:5} images={len(imgs):<6} labels={len(lbls):<6} "
              f"{'OK' if imgs and len(imgs)==len(lbls) else 'MISMATCH - fix before training'}")

        counts = collections.Counter()
        for f in lbls:
            for line in f.read_text().splitlines():
                if line.strip():
                    counts[int(line.split()[0])] += 1
        for cid, n in sorted(counts.items()):
            name = cfg["names"][cid] if isinstance(cfg["names"], list) else cfg["names"].get(cid, cid)
            flag = "  <-- too few to learn" if n < 100 else ""
            print(f"      {str(name):<16} {n:>6}{flag}")
        print()


### Before anything else: is the split honest?

A model evaluated on images it trained on will score brilliantly and be worthless.
This is not a hypothetical — the first run of this notebook used `coco128`, whose
`train` and `val` keys point at **the same 128 images**. Every metric jumped by
0.2 and all of it was memorisation.

The failure is dangerous precisely because it looks like spectacular success. Run
this before you trust any number below.

In [ ]:
import yaml

def check_split(data_yaml):
    """Refuse to proceed if val is the same data as train."""
    if data_yaml.endswith("coco128.yaml"):
        print("LEAKED: coco128 uses the same 128 images for train and val.")
        print("        It is a pipeline smoke test only. Any metric from it is")
        print("        memorisation, not accuracy. Swap in a real dataset.")
        return False

    cfg = yaml.safe_load(open(data_yaml))
    train, val = str(cfg.get("train", "")), str(cfg.get("val", ""))
    if not val:
        print("NO VAL SPLIT: nothing to measure against. Add one.")
        return False
    if train == val:
        print(f"LEAKED: train and val are both '{train}'.")
        print("        Split your data before training, or the numbers are fiction.")
        return False

    print(f"OK: train='{train}'  val='{val}'  (different)")
    return True

SPLIT_OK = check_split(DATA_YAML)
print("\nMetrics below are trustworthy." if SPLIT_OK
      else "\nMetrics below are NOT trustworthy. Fix the split first.")


## 4. Train

Settings that matter, and why:

- **`model="yolo11s.pt"`** — start from the weights you already serve, not from
  scratch. Random init on a small dataset produces a worse model than the
  pretrained one you began with.
- **`flipud=0.0`** — traffic is never upside down. Vertical flips teach the model
  a situation that does not exist.
- **`degrees=5.0`** — cameras are roughly level; large rotations are noise.
- **`patience=25`** — stop when validation stops improving rather than burning
  GPU time on overfitting.
- **`epochs=100`** — an upper bound, not a target. Early stopping usually ends it
  sooner.

Roughly an hour on a T4 for a few thousand images. Colab disconnects idle
sessions, so keep the tab open.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,
    patience=25,
    name="traffic-yolo",
    pretrained=True,
    fliplr=0.5,      # mirroring a road scene is valid
    flipud=0.0,      # flipping it upside down is not
    degrees=5.0,     # cameras sit roughly level
    scale=0.5,
    mosaic=1.0,
)
print("run dir:", results.save_dir)


## 5. Did it actually help?

This is the cell people skip, and it is the only one that answers the question.
A fine-tune on too little data, or too few epochs, produces a model **worse** than
the stock weights — I measured exactly that while validating this notebook: two
epochs on 128 images took `car` mAP50 from 0.442 down to 0.212.

Compare against the stock baseline on the same data. If your numbers are not
better, do not ship the model.

In [ ]:
stock = YOLO("yolo11s.pt").val(data=DATA_YAML, imgsz=640, device=0, verbose=False)
tuned = YOLO(f"{results.save_dir}/weights/best.pt").val(data=DATA_YAML, imgsz=640, device=0, verbose=False)

print(f"{'metric':<12} {'stock':>8} {'tuned':>8} {'change':>9}")
print("-" * 41)
for label, a, b in [
    ("mAP50",    stock.box.map50, tuned.box.map50),
    ("mAP50-95", stock.box.map,   tuned.box.map),
    ("precision",stock.box.mp,    tuned.box.mp),
    ("recall",   stock.box.mr,    tuned.box.mr),
]:
    arrow = "better" if b > a else "WORSE"
    print(f"{label:<12} {a:>8.3f} {b:>8.3f} {b-a:>+9.3f}  {arrow}")

if not SPLIT_OK:
    print("\n*** These numbers are measured on training data. They are")
    print("*** memorisation, not accuracy. Do not publish or ship them. ***")

print("\nPer-class (the car row is the one you care about):")
for i, cid in enumerate(tuned.box.ap_class_index):
    print(f"  class {cid}: R={tuned.box.r[i]:.3f}  mAP50={tuned.box.ap50[i]:.3f}")


## 6. Download the weights

Save `best.pt` and put it in `weights/` in the repo.

In [ ]:
from google.colab import files
files.download(f"{results.save_dir}/weights/best.pt")


## 7. Serve it

Rename the file to something meaningful, drop it in `weights/`, and point the
service at it. No other code changes — `MODEL_PATH` is read from the environment
and the class filter falls back to every class the model knows when its labels
do not match COCO's names.

```bash
MODEL_PATH=weights/traffic-yolo.pt uvicorn app.main:app --port 7860
```

Then re-measure locally and paste the new table into the README's Accuracy
section, replacing the coco128 figures:

```bash
MODEL_PATH=weights/traffic-yolo.pt python scripts/evaluate.py --data dataset/data.yaml
```

Weights are gitignored — they are too large for GitHub. Publish them as a
Hugging Face model repo and have the container download them at build time, or
commit them via LFS if they are small enough.

**If you trained on different classes than COCO**, update `TRAFFIC_CLASS_NAMES`
and `CLASS_COLORS` in `app/config.py` to match your `data.yaml`, or the console's
filter chips will not line up with what the model predicts.